# Week 3, Classification: Counting with Weights

**What you'll do today.** You'll train a transparent classifier, a logistic regression, on a
pre-labeled pop corpus, then **read its mind**: the signed weights that are its most positive
and most negative words.

In [ ]:
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

In [ ]:
# First, install the few packages Colab doesn't already ship. If you opened this
# notebook with the whole repo, the line below uses our pinned versions:
%pip install -q -r requirements.txt -c constraints.txt

# Opened this notebook on its own, without the repo files? Comment the line above
# and use this explicit pinned install instead:
# %pip install -q pandas scikit-learn matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
print("imports ok")

In [ ]:
# Are we running tiny + offline (the compatibility harness sets this), or in a real
# Colab class session? Everything below adapts so the notebook runs either way.
import os
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode (tiny, offline):", SMOKE)

## A pre-labeled corpus

In class this is ~1,000 rows, AITA posts tagged YTA/NTA, or film reviews tagged pos/neg.

In [ ]:
pos = [
    "a stunning and heartfelt triumph", "brilliant performances and a gorgeous script",
    "i loved every minute, funny and moving", "a delightful, clever, wonderful film",
    "beautiful, touching, and genuinely surprising", "superb acting, a masterpiece",
    "charming and uplifting from start to finish", "an instant classic, joyful and warm",
]
neg = [
    "a boring and predictable mess", "dull, lifeless, and far too long",
    "i hated it, a tedious waste of time", "awful script and wooden acting",
    "a disappointing, forgettable bore", "clumsy, ugly, and painfully slow",
    "terrible pacing and a stupid ending", "a dreadful, joyless slog",
]
df = pd.DataFrame({"text": pos + neg, "label": ["pos"]*len(pos) + ["neg"]*len(neg)})
print(df.sample(4, random_state=0).to_string(index=False))

## Train it, the AI writes this; reading the result is your job

Each word becomes a column, the model learns a weight (a vote, for or against), and it adds them
up.

In [ ]:
vec = CountVectorizer()
X = vec.fit_transform(df["text"])
y = (df["label"] == "pos").astype(int)

# Tiny data, so train on all of it; with a real corpus you'd hold out a test set like this:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("held-out accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))
clf_full = LogisticRegression(max_iter=1000).fit(X, y)  # refit on all for weight-reading

## Read its mind, the signed weights

The most **positive** and most **negative** words are the model's mind on the table.

In [ ]:
terms = np.array(vec.get_feature_names_out())
w = clf_full.coef_.ravel()
order = w.argsort()
print("Most NEGATIVE words:", ", ".join(terms[order[:6]]))
print("Most POSITIVE words:", ", ".join(terms[order[-6:][::-1]]))

plt.figure(figsize=(6,3))
top = np.concatenate([order[:6], order[-6:]])
plt.barh(terms[top], w[top])
plt.title("The classifier's signed weights")
plt.tight_layout(); plt.show()

### Build it a "black cat"

The Teachable Machine demo learned a bias from a skewed training set (orange cats, brown dogs).

In [ ]:
def predict(s):
    p = clf_full.predict_proba(vec.transform([s]))[0, 1]
    return f"{'pos' if p>0.5 else 'neg'} (p_pos={p:.2f})"

print("clear positive :", predict("a wonderful, brilliant film"))
print("clear negative :", predict("a boring, awful mess"))
print("the black cat  :", predict("not bad at all, I did not dislike it"))  # negation it can't see

### Delight beat, what says "this reviewer hated it"?

Just enjoy it: the little cultural fingerprint the machine learned.

## Where this goes

You built the **transparent** reader: cheap, and you can see exactly what it weighed.

**Sketch:** point the AI at a labeled dataset you're curious about, train a quick logistic
regression, and screenshot its five most positive and five most negative words.